# Why the midpoint is enough

Lengths in the table are $1$, $2$, and $5$. Consider every threshold $t$ with

$$
2 \le t < 5
$$

A part with length $2$ satisfies $2 \le t$, so it goes left under the rule "length $\le t$". A part with length $5$ satisfies $5 > t$, so it goes right. No training part has a length strictly between $2$ and $5$. So every such $t$ builds the same two groups.

The midpoint $7/2$ is one such $t$. So is $3$, and so is $4.9$. They are the same question as far as these eight parts are concerned.

You do not need to try $t = 100$. Everything would go left, the right child would be empty, and there would be no split.


In [1]:
# Every threshold from 2 up to, but not including, 5 builds one partition.
from fractions import Fraction

lengths = [1, 1, 2, 2, 1, 2, 5, 5]
labels = ["pass", "pass", "pass", "pass", "fail", "fail", "fail", "fail"]

def groups(threshold):
    left = [label for length, label in zip(lengths, labels) if length <= threshold]
    right = [label for length, label in zip(lengths, labels) if length > threshold]
    return left, right

assert groups(Fraction(7, 2)) == groups(3) == groups(Fraction(49, 10))
left, right = groups(Fraction(7, 2))
assert left.count("pass") == 4 and left.count("fail") == 2
assert right == ["fail", "fail"]
# A threshold below every larger length but not inside the gap can differ.
assert groups(1) != groups(Fraction(7, 2))
print("left", left)
print("right", right)


left ['pass', 'pass', 'pass', 'pass', 'fail', 'fail']
right ['fail', 'fail']


## The tie-break, written down

Length $\le 7/2$ and weight $\le 3$ both gain $1/6$. The rule used in the rest of this path:

1. Prefer the larger gain.
2. If the gains match, prefer the earlier feature. Length is listed before weight.
3. If the same feature has two equal cuts, prefer the smaller threshold.

Rule 2 selects length $\le 7/2$ at the root. The weight question is still available underneath.


In [2]:
# The tie-break is a sort key: gain, then feature order, then threshold.
from fractions import Fraction

candidates = [
    (Fraction(1, 30), 0, Fraction(3, 2)),
    (Fraction(1, 6), 0, Fraction(7, 2)),
    (Fraction(1, 30), 1, Fraction(3, 2)),
    (Fraction(1, 6), 1, 3),
]
# Max gain, then smaller feature index, then smaller threshold.
winner = max(candidates, key=lambda item: (item[0], -item[1], -item[2]))
assert winner == (Fraction(1, 6), 0, Fraction(7, 2))
print("winner feature 0 threshold", winner[2])


winner feature 0 threshold 7/2


## A pitfall

The midpoint can look like a fitted constant. It is only a label for a gap. If a new part arrives with length $3$, it falls on the left of $7/2$ even though no training part had length $3$. Say that when you report the tree. The question generalizes. The midpoint did not estimate a physical length of $3.5\,\text{cm}$ beyond the fact that the gap is between $2$ and $5$.

## What to carry forward

One candidate per gap between consecutive training values is enough, because the gap contains no training part. Ties break toward length, then toward the smaller cut. The root question is length $\le 7/2$.
